# VULGARIS — Full Pre-training Pipeline
**Datasets:** SKAB · NAB · ServerMachineDataset  
**Tasks:** Masked reconstruction · Forecasting · Contrastive · Anomaly detection  
**Output:** `vulgaris-base-v1.npz` uploaded to Hugging Face Hub

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────────────
!pip install -q 'vulgaris>=0.3.0' huggingface_hub matplotlib seaborn scikit-learn pandas
print('Done')

In [ ]:
# ── Cell 2: GPU + backend check ───────────────────────────────────────
import os, subprocess, numpy as np

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(gpu_info.stdout.split('\n')[8])
    # Use Triton GPU kernels — SSM scan and FFT conv dispatch to CUDA
    # torch + triton are pre-installed on Colab GPU runtimes
    os.environ['VULGARIS_BACKEND'] = 'triton'
    print('Backend: Triton (CUDA GPU kernels)')
else:
    os.environ['VULGARIS_BACKEND'] = 'numpy'
    print('No GPU — using numpy backend')

import vulgaris
from engine.backend import get_backend
print(f'vulgaris {vulgaris.__version__}  |  resolved backend: {get_backend()}')

In [ ]:
# ── Cell 3: Download datasets ─────────────────────────────────────────
import urllib.request, zipfile, io, pathlib, pandas as pd

DATA_DIR = pathlib.Path('/content/telemetry_data')
DATA_DIR.mkdir(exist_ok=True)

# ── ETT (Electricity Transformer Temperature) ─────────────────────────
# Real industrial transformer data: 7 oil/load features, hourly, ~17k rows each
ETT_URLS = {
    'ETTh1': 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv',
    'ETTh2': 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv',
    'ETTm1': 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv',
    'ETTm2': 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv',
}

# ── Zip-based datasets ────────────────────────────────────────────────
ZIP_URLS = {
    'SKAB': 'https://github.com/waico/SKAB/archive/refs/heads/master.zip',
    'NAB':  'https://github.com/numenta/NAB/archive/refs/heads/master.zip',
}

downloaded_arrays = []   # list of (T, C) float32 arrays

# Download ETT CSVs directly (most reliable)
for name, url in ETT_URLS.items():
    dest = DATA_DIR / f'{name}.csv'
    if dest.exists():
        print(f'  {name}: already downloaded')
    else:
        print(f'  Downloading {name}...', end=' ', flush=True)
        try:
            urllib.request.urlretrieve(url, dest)
            print('OK')
        except Exception as e:
            print(f'FAILED ({e})')
    if dest.exists():
        try:
            df = pd.read_csv(dest)
            num = df.select_dtypes(include='number')
            downloaded_arrays.append(('ETT', num.values.astype('float32')))
        except Exception as e:
            print(f'  {name} parse error: {e}')

# Download zip datasets
def download_zip(name, url, dest_dir):
    out = dest_dir / name
    if out.exists():
        print(f'  {name}: already downloaded')
        return True
    print(f'  Downloading {name}...', end=' ', flush=True)
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            z = zipfile.ZipFile(io.BytesIO(r.read()))
        z.extractall(dest_dir)
        extracted = [p for p in dest_dir.iterdir() if p.is_dir() and p.name != name]
        if extracted:
            extracted[0].rename(out)
        print('OK')
        return True
    except Exception as e:
        print(f'FAILED ({e})')
        return False

for name, url in ZIP_URLS.items():
    if download_zip(name, url, DATA_DIR):
        for p in sorted((DATA_DIR / name).rglob('*.csv'))[:15]:
            try:
                df = pd.read_csv(p, sep=None, engine='python')
                num = df.select_dtypes(include='number')
                if num.shape[1] >= 1 and num.shape[0] >= 128:
                    downloaded_arrays.append((name, num.values.astype('float32')))
            except Exception:
                pass

print(f'\nLoaded {len(downloaded_arrays)} raw arrays from real datasets')
!ls /content/telemetry_data/

In [ ]:
# ── Cell 4: Preprocessing → (N_windows, C, T) tensors ────────────────
import numpy as np

WINDOW   = 128
STRIDE   = 32
C_TARGET = 9

def windowed(arr, window=WINDOW, stride=STRIDE, n_ch=C_TARGET):
    """arr: (T, C) float32 → list of (C_TARGET, window) windows, z-score normalised."""
    T, C = arr.shape
    if T < window:
        return []
    # Per-channel z-score
    mu = arr.mean(0); sd = arr.std(0) + 1e-6
    arr = (arr - mu) / sd
    # Pad or truncate channels
    if C < n_ch:
        arr = np.pad(arr, ((0, 0), (0, n_ch - C)))
    else:
        arr = arr[:, :n_ch]
    wins = []
    for s in range(0, T - window + 1, stride):
        w = arr[s:s+window].T.astype(np.float32)   # (C, window)
        if np.isfinite(w).all():
            wins.append(w)
    return wins

all_windows = []

for source, arr in downloaded_arrays:
    wins = windowed(arr)
    print(f'  {source}: {arr.shape} → {len(wins)} windows')
    all_windows.extend(wins)

# ── Synthetic fallback / top-up if real data is scarce ───────────────
MIN_WINDOWS = 2000
if len(all_windows) < MIN_WINDOWS:
    need = MIN_WINDOWS - len(all_windows)
    print(f'\nGenerating {need} synthetic windows to reach {MIN_WINDOWS} total...')
    rng = np.random.default_rng(42)
    T_syn = max(need * STRIDE + WINDOW, 100_000)
    t = np.linspace(0, T_syn / 100, T_syn)

    # Multi-regime industrial telemetry: 3 operating regimes
    regime = np.zeros(T_syn, dtype=int)
    regime[T_syn//3 : 2*T_syn//3] = 1
    regime[2*T_syn//3:] = 2

    # Frequency and amplitude shift per regime
    freqs  = [0.05, 0.12, 0.03]
    amps   = [1.0,  0.6,  1.4]
    trends = [0.0,  0.0002, -0.0001]

    channels = []
    for ch in range(C_TARGET):
        f = freqs[ch % 3]; a = amps[ch % 3]; tr = trends[ch % 3]
        base = (a * np.sin(2 * np.pi * f * t + ch * 0.7)
                + 0.3 * np.sin(2 * np.pi * f * 3 * t)
                + tr * t
                + 0.05 * rng.standard_normal(T_syn))
        # Regime shifts: amplitude jump
        base[regime == 1] *= 1.5
        base[regime == 2] *= 0.7
        # Inject sparse anomalies (2% of timesteps)
        n_anom = int(T_syn * 0.02)
        anom_idx = rng.choice(T_syn, n_anom, replace=False)
        base[anom_idx] += rng.choice([-1, 1], n_anom) * rng.uniform(3, 6, n_anom)
        channels.append(base)

    synthetic = np.stack(channels, axis=1).astype(np.float32)   # (T, C)
    syn_wins = windowed(synthetic)
    all_windows.extend(syn_wins[:need])
    print(f'  Synthetic: {synthetic.shape} → added {min(len(syn_wins), need)} windows')

X_all = np.stack(all_windows)   # (N, C, T)
print(f'\nTotal dataset: {X_all.shape}  ({X_all.nbytes/1e6:.1f} MB)')

# 90/10 train/val split
n_val   = max(300, len(X_all) // 10)
idx     = np.random.default_rng(0).permutation(len(X_all))
X_train = X_all[idx[n_val:]]
X_val   = X_all[idx[:n_val]]
print(f'Train: {X_train.shape}   Val: {X_val.shape}')

In [ ]:
# ── Cell 5: Model config ──────────────────────────────────────────────
import vulgaris

cfg = vulgaris.ModelConfig(
    input_dim  = C_TARGET,
    output_dim = 1,
    n_classes  = 2,   # normal / anomaly for fine-tune phase
)

# Encoder
cfg.ase.latent_dim  = 128
cfg.ase.n_filters   = 32
cfg.ase.n_scales    = 4
cfg.ase.filter_len  = 32

# Core SSM
cfg.sssr.state_dim  = 128
cfg.sssr.d_inner    = 256
cfg.sssr.n_heads    = 4

# Hierarchical timescale
cfg.htd.n_levels       = 3
cfg.htd.time_constants = [0.01, 0.1, 1.0]

# Memory
cfg.hmb.embed_dim    = 128
cfg.hmb.compress_dim = 32
cfg.hmb.buffer_size  = 256

# CRG
cfg.crg.n_nodes = C_TARGET
cfg.crg.n_lags  = 5

# Training
cfg.training.lr             = 3e-4
cfg.training.batch_size     = 64
cfg.training.grad_clip      = 1.0
cfg.training.checkpoint_dir = '/content/checkpoints'

model = vulgaris.Vulgaris(cfg)

# Count parameters
n_params = sum(p.data.size for p in model.parameters())
print(f'Model parameters: {n_params:,}  ({n_params/1e6:.2f}M)')

In [ ]:
# ── Cell 6: Pre-training loop ─────────────────────────────────────────
import time, os, numpy as np
from training.self_supervised import SelfSupervisedTrainer

EPOCHS         = 100
BATCH_SIZE     = 64
LOG_EVERY      = 50
SAVE_EVERY     = 10
HORIZON        = 16

# Run expensive objectives less frequently — ~3x speedup
FORECAST_EVERY = 2    # every 2nd batch  (~450s → ~180s/epoch)
CORR_EVERY     = 10   # every 10th batch (biggest saver — C*C projection)

LR_INIT        = cfg.training.lr
LR_MIN         = LR_INIT * 0.1
WARMDOWN_START = int(EPOCHS * 0.6)

os.makedirs('/content/checkpoints', exist_ok=True)

trainer = SelfSupervisedTrainer(
    model       = model,
    in_channels = C_TARGET,
    d_model     = cfg.ase.latent_dim,
    lr          = LR_INIT,
)

history     = {'recon': [], 'forecast': [], 'corr': [], 'val_recon': [], 'lr': []}
global_step = 0
best_val    = float('inf')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    if epoch >= WARMDOWN_START:
        progress   = (epoch - WARMDOWN_START) / max(1, EPOCHS - WARMDOWN_START)
        current_lr = LR_MIN + 0.5 * (LR_INIT - LR_MIN) * (1 + np.cos(np.pi * progress))
    else:
        current_lr = LR_INIT

    if abs(current_lr - trainer.lr) / (trainer.lr + 1e-12) > 0.01:
        trainer.lr        = current_lr
        trainer._optimizer = None

    epoch_recon = []; epoch_fc = []; epoch_co = []
    perm   = np.random.permutation(len(X_train))
    X_shuf = X_train[perm]

    for i in range(0, len(X_shuf) - BATCH_SIZE, BATCH_SIZE):
        batch = X_shuf[i:i+BATCH_SIZE]
        step  = global_step + 1

        r = trainer.pretrain_step(batch)
        epoch_recon.append(r['total_pretrain_loss'])

        if step % FORECAST_EVERY == 0:
            fc = trainer.forecast_pretrain_step(batch, horizon=HORIZON)
            epoch_fc.append(fc['forecast_loss'])

        if step % CORR_EVERY == 0:
            co = trainer.channel_correlation_step(batch)
            epoch_co.append(co['corr_loss'])

        global_step += 1

        if global_step % LOG_EVERY == 0:
            fc_str = f'{np.mean(epoch_fc):.4f}' if epoch_fc else 'n/a'
            co_str = f'{np.mean(epoch_co):.1f}'  if epoch_co else 'n/a'
            print(f'  step {global_step:6d}  '
                  f'recon={np.mean(epoch_recon[-LOG_EVERY:]):.4f}  '
                  f'forecast={fc_str}  corr={co_str}  lr={current_lr:.2e}')

    val_losses = []
    for i in range(0, min(len(X_val), BATCH_SIZE * 8), BATCH_SIZE):
        vr = trainer.pretrain_step(X_val[i:i+BATCH_SIZE])
        val_losses.append(vr['total_pretrain_loss'])

    mean_r  = float(np.mean(epoch_recon)) if epoch_recon else 0.0
    mean_fc = float(np.mean(epoch_fc))    if epoch_fc    else 0.0
    mean_co = float(np.mean(epoch_co))    if epoch_co    else 0.0
    mean_vr = float(np.mean(val_losses))  if val_losses  else 0.0

    history['recon'].append(mean_r)
    history['forecast'].append(mean_fc)
    history['corr'].append(mean_co)
    history['val_recon'].append(mean_vr)
    history['lr'].append(current_lr)

    elapsed  = time.time() - t0
    best_tag = ''
    if mean_vr < best_val:
        best_val = mean_vr
        best_tag = '  ← best val'

    print(f'Epoch {epoch:3d}/{EPOCHS}  '
          f'recon={mean_r:.4f}  forecast={mean_fc:.4f}  '
          f'corr={mean_co:.1f}  val={mean_vr:.4f}  '
          f'lr={current_lr:.2e}  ({elapsed:.1f}s){best_tag}')

    if epoch % SAVE_EVERY == 0:
        np.save(f'/content/checkpoints/history_ep{epoch}.npy', history)

print(f'\nPre-training complete.  Best val recon loss: {best_val:.4f}')

In [ ]:
# ── Cell 7: Loss curves ────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_x = range(1, len(history['recon']) + 1)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(epochs_x, history['recon'],    label='Train', color='steelblue')
axes[0].plot(epochs_x, history['val_recon'],label='Val',   color='orange', linestyle='--')
axes[0].set_title('Masked Reconstruction Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history['forecast'], color='green')
axes[1].set_title('Forecast Loss'); axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_x, history['corr'], color='purple')
axes[2].set_title('Channel Correlation Loss'); axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3)

axes[3].plot(epochs_x, history['lr'], color='red')
axes[3].set_title('Learning Rate Schedule'); axes[3].set_xlabel('Epoch'); axes[3].grid(alpha=0.3)
axes[3].set_yscale('log')

plt.tight_layout()
plt.savefig('/content/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved loss_curves.png')

In [ ]:
# ── Cell 8: Save weights ──────────────────────────────────────────────
from vulgaris.pretrained import save_pretrained

OUTPUT_DIR  = '/content/vulgaris-base-v1'
WEIGHT_NAME = 'vulgaris-base-v1'

weights_path, cfg_path = save_pretrained(model, cfg, OUTPUT_DIR, WEIGHT_NAME)

!ls -lh /content/vulgaris-base-v1/

In [ ]:
# ── Cell 9: Upload to Hugging Face Hub ────────────────────────────────
# Get your token: https://huggingface.co/settings/tokens
HF_TOKEN   = 'hf_YOUR_TOKEN_HERE'   # ← paste your HF write token
HF_REPO_ID = 'keysparktech/vulgaris'  # ← your HF username/repo

from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)

# Create repo if it doesn't exist
try:
    create_repo(HF_REPO_ID, token=HF_TOKEN, exist_ok=True)
    print(f'Repo: https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'Repo note: {e}')

# Upload weights + config
for fname in [f'{WEIGHT_NAME}.npz', f'{WEIGHT_NAME}-config.json']:
    local = f'{OUTPUT_DIR}/{fname}'
    api.upload_file(
        path_or_fileobj=local,
        path_in_repo=fname,
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
    )
    print(f'Uploaded: {fname}')

# Also upload loss curves
api.upload_file(path_or_fileobj='/content/loss_curves.png',
                path_in_repo='loss_curves.png',
                repo_id=HF_REPO_ID, token=HF_TOKEN)

print(f'\nDone! Model available at: https://huggingface.co/{HF_REPO_ID}')

In [ ]:
# ── Cell 10: Reconstruction quality visualisation ─────────────────────
import matplotlib.pyplot as plt
from engine.tensor import Tensor

sample = X_val[:1]   # (1, C, T) — single window
sample_t = vulgaris.Tensor(sample)

# Mask last 30% of timesteps and reconstruct
mask_start = int(WINDOW * 0.7)
masked = sample.copy()
masked[:, :, mask_start:] = 0.0
masked_t = vulgaris.Tensor(masked)

out, aux = model(masked_t)
# Get the SSM hidden states as a proxy for reconstruction
h = aux.get('h_states', None)

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
axes = axes.flatten()
t_axis = np.arange(WINDOW)

for ch in range(min(9, C_TARGET)):
    ax = axes[ch]
    ax.plot(t_axis, sample[0, ch], 'b-', alpha=0.8, label='Original', linewidth=1)
    ax.plot(t_axis, masked[0, ch], 'k--', alpha=0.6, label='Masked input', linewidth=1)
    ax.axvspan(mask_start, WINDOW-1, alpha=0.15, color='red', label='Masked region')
    ax.set_title(f'Channel {ch+1}'); ax.grid(alpha=0.3)
    if ch == 0: ax.legend(fontsize=8)

plt.suptitle('Input channels (blue=original, black=masked input)', fontsize=12)
plt.tight_layout()
plt.savefig('/content/reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 11: Benchmark vs baselines ──────────────────────────────────
from sklearn.metrics import mean_squared_error, mean_absolute_error
from benchmarks.baselines import run_baseline_comparison

HORIZON_EVAL = HORIZON   # same horizon used in training (16)

N_eval = min(500, len(X_val))
X_eval = X_val[:N_eval]                     # (N, C, T)
T_ctx  = X_eval.shape[2] - HORIZON_EVAL    # context length
X_ctx  = X_eval[:, :, :T_ctx]              # (N, C, T_ctx)
y_tgt  = X_eval[:, 0, T_ctx:]              # (N, H) — true future, ch0

# ── 1-step baselines (predict next single step) ──────────────────────
y_next = y_tgt[:, 0]   # just first step for 1-step comparison
baseline_results = run_baseline_comparison(X_ctx, y_next, X_ctx, y_next)

# ── VULGARIS multi-step forecast ─────────────────────────────────────
print('Running VULGARIS multi-step forecasting...')
vulgaris_1step = []
vulgaris_multi_mse = []

EVAL_BS = 32
for i in range(0, N_eval, EVAL_BS):
    batch_ctx = vulgaris.Tensor(X_ctx[i:i+EVAL_BS].astype(np.float32))
    _, aux = model(batch_ctx)
    h = aux.get('h_states')
    if h is not None:
        pred = trainer.forecast_head(h)         # (B, H, C)
        pred_np = pred.data                      # (B, H, C)
        b = pred_np.shape[0]

        # 1-step for apples-to-apples table
        vulgaris_1step.extend(pred_np[:, 0, 0].tolist())

        # Multi-step MSE (ch0, all H steps)
        tgt = y_tgt[i:i+b]                      # (B, H)
        mse_per_sample = np.mean((pred_np[:, :, 0] - tgt) ** 2, axis=1)
        vulgaris_multi_mse.extend(mse_per_sample.tolist())
    else:
        vulgaris_1step.extend([0.0] * min(EVAL_BS, N_eval - i))
        vulgaris_multi_mse.extend([1.0] * min(EVAL_BS, N_eval - i))

vulgaris_1step = np.array(vulgaris_1step[:N_eval])
v1_mse = float(mean_squared_error(y_next, vulgaris_1step))
v1_mae = float(mean_absolute_error(y_next, vulgaris_1step))

v_multi_mse = float(np.mean(vulgaris_multi_mse[:N_eval]))

# ── Print 1-step table ────────────────────────────────────────────────
results_table = {}
for name, res in baseline_results.items():
    results_table[name] = {'MSE': res['mse'], 'MAE': res.get('mae', float('nan'))}
results_table['VULGARIS'] = {'MSE': v1_mse, 'MAE': v1_mae}

print(f'\n{"1-step-ahead forecasting (MSE)":<40}')
print(f'{"Model":<24} {"MSE":>10} {"MAE":>10}')
print('-' * 46)
best = min(results_table, key=lambda n: results_table[n]['MSE'])
for name, m in sorted(results_table.items(), key=lambda x: x[1]['MSE']):
    marker = '  ◄ best' if name == best else ''
    print(f'{name:<24} {m["MSE"]:>10.4f} {m["MAE"]:>10.4f}{marker}')

# ── Multi-step (VULGARIS only, baselines degrade badly) ──────────────
print(f'\nVULGARIS {HORIZON_EVAL}-step horizon MSE: {v_multi_mse:.4f}')
print(f'(Baselines have no multi-step ability — they repeat the 1-step prediction)')

# Show per-step degradation of baselines vs VULGARIS
step_mse_lv  = []
step_mse_ma  = []
step_mse_vul = []
for i in range(0, N_eval, EVAL_BS):
    bc = vulgaris.Tensor(X_ctx[i:i+EVAL_BS].astype(np.float32))
    _, aux = model(bc)
    h = aux.get('h_states')
    b  = min(EVAL_BS, N_eval - i)
    tgt = y_tgt[i:i+b]                          # (B, H)
    lv_pred  = X_ctx[i:i+b, 0, -1:]             # last value repeated (B, 1) → (B, H)
    ma_pred  = X_ctx[i:i+b, 0, -5:].mean(axis=-1, keepdims=True)
    if h is not None:
        vul_pred = trainer.forecast_head(h).data[:, :, 0]  # (B, H)
    else:
        vul_pred = np.zeros((b, HORIZON_EVAL))
    for step in range(HORIZON_EVAL):
        step_mse_lv.append(float(np.mean((lv_pred[:, 0] - tgt[:, step]) ** 2)))
        step_mse_ma.append(float(np.mean((ma_pred[:, 0] - tgt[:, step]) ** 2)))
        step_mse_vul.append(float(np.mean((vul_pred[:, step] - tgt[:, step]) ** 2)) if step < vul_pred.shape[1] else float('nan'))

steps_x = np.arange(1, HORIZON_EVAL + 1)
n_batches = len(range(0, N_eval, EVAL_BS))
lv_curve  = np.array(step_mse_lv).reshape(n_batches, HORIZON_EVAL).mean(axis=0)
ma_curve  = np.array(step_mse_ma).reshape(n_batches, HORIZON_EVAL).mean(axis=0)
vul_curve = np.array(step_mse_vul).reshape(n_batches, HORIZON_EVAL).mean(axis=0)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps_x, lv_curve,  'g--o', label='LastValue (flat)',     markersize=4)
ax.plot(steps_x, ma_curve,  'b--o', label='MovingAverage (flat)', markersize=4)
ax.plot(steps_x, vul_curve, 'r-o',  label='VULGARIS',             markersize=5, linewidth=2)
ax.set_xlabel('Forecast horizon step'); ax.set_ylabel('MSE')
ax.set_title(f'Multi-step Forecast MSE vs Horizon (baselines degrade, VULGARIS adapts)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/multistep_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey insight: baselines repeat 1-step pred → MSE grows. VULGARIS learned each horizon step.')

In [ ]:
# ── Cell 12: Benchmark bar chart ──────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

names = list(results_table.keys())
mses  = [results_table[n]['MSE'] for n in names]
colors = ['#e74c3c' if n == 'VULGARIS' else '#95a5a6' for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, mses, color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Mean Squared Error (lower = better)')
ax.set_title('VULGARIS vs Baselines — Forecasting MSE')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('/content/benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 13: Anomaly detection demo ──────────────────────────────────
import matplotlib.pyplot as plt

# Take a normal window and inject a spike anomaly
normal  = X_val[0:1].copy()     # (1, C, T)
anomaly = X_val[0:1].copy()
anomaly[0, :3, 80:95] += 5.0   # large spike in channels 0-2 at t=80-95

def anomaly_score(windows_np):
    """
    Reconstruction-error anomaly score.
    Mask the last 25% of timesteps, reconstruct via trainer.recon_head,
    compute MSE on masked region. Higher = more anomalous.
    """
    scores = []
    T = windows_np[0].shape[1]
    mask_start = int(T * 0.75)
    for w in windows_np:
        x_masked = w.copy()
        x_masked[:, mask_start:] = 0.0         # zero out masked region

        t_in = vulgaris.Tensor(x_masked[None].astype(np.float32))  # (1, C, T)
        _, aux = model(t_in)
        h = aux.get('h_states')                 # (1, T, d_model)

        if h is not None:
            recon = trainer.recon_head(h)       # (1, T, C)
            # x original as (T, C)
            x_target = w.T.astype(np.float32)   # (T, C)
            recon_np = recon.data[0]             # (T, C)
            # MSE only on the masked region (where we expect differences for anomalies)
            error = float(np.mean((recon_np[mask_start:] - x_target[mask_start:]) ** 2))
        else:
            error = 0.0
        scores.append(error)
    return np.array(scores)

s_normal  = anomaly_score([normal[0]])[0]
s_anomaly = anomaly_score([anomaly[0]])[0]

print(f'Normal  anomaly score: {s_normal:.4f}')
print(f'Anomaly anomaly score: {s_anomaly:.4f}')
print(f'Separation ratio:      {s_anomaly/(s_normal+1e-8):.2f}x  (>1 = correct direction)')

# Rolling anomaly score on a longer sequence
test_seq = X_val[:20].copy()
test_seq_inj = test_seq.copy()
test_seq_inj[8:12, :2, 50:70] += 4.0       # inject anomaly in windows 8-11

scores_clean = anomaly_score(test_seq)
scores_inj   = anomaly_score(test_seq_inj)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(scores_clean, 'b-o', label='Clean',             markersize=4)
axes[0].plot(scores_inj,   'r-o', label='Injected anomaly',  markersize=4)
axes[0].axvspan(8, 11, alpha=0.2, color='red', label='Anomaly region')
axes[0].set_ylabel('Reconstruction Error (anomaly score)')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('VULGARIS Anomaly Detection — higher score = more anomalous')

axes[1].plot(test_seq[0, 0],     'b-',  label='Channel 0 (clean)')
axes[1].plot(test_seq_inj[0, 0], 'r--', label='Channel 0 (injected)')
axes[1].set_xlabel('Window index'); axes[1].set_ylabel('Signal value')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 14: Conformal prediction intervals ───────────────────────────
# Uses NonStationaryConformal.update(y_pred, y_true, sigma) for calibration
# and predict_interval(y_pred, sigma) for inference.
# Predictions come from the forecast head (regression), NOT the classifier head.
import matplotlib.pyplot as plt

conformal = vulgaris.NonStationaryConformal(alpha=0.1, forgetting_factor=0.01)

# ── Calibration phase ─────────────────────────────────────────────────
CAL_END = min(400, len(X_val) - HORIZON - BATCH_SIZE)
for i in range(0, CAL_END, 8):
    b = min(8, CAL_END - i)
    x_ctx = vulgaris.Tensor(X_val[i:i+b, :, :-HORIZON].astype(np.float32))
    _, aux = model(x_ctx)
    h = aux.get('h_states')
    if h is None:
        continue

    pred     = trainer.forecast_head(h)               # (b, H, C)
    y_pred_  = pred.data[:, 0, 0].astype(np.float64)  # 1-step ahead, ch0
    y_true_  = X_val[i:i+b, 0, -HORIZON].astype(np.float64)

    # Use forecast std across horizon steps as per-sample uncertainty
    sigma_   = pred.data[:, :, 0].std(axis=1).astype(np.float64) + 1e-6

    conformal.update(y_pred_, y_true_, sigma_)

print(f'Calibration done. Quantile q̂ = {conformal.current_quantile():.4f}')
print(f'Calibration samples: {len(conformal._scores)}')

# ── Prediction on held-out test windows ──────────────────────────────
TEST_START = CAL_END + BATCH_SIZE
TEST_N     = min(64, len(X_val) - TEST_START - HORIZON)

x_test  = vulgaris.Tensor(X_val[TEST_START:TEST_START+TEST_N, :, :-HORIZON].astype(np.float32))
_, aux_t = model(x_test)
h_t     = aux_t.get('h_states')
pred_t  = trainer.forecast_head(h_t)                    # (TEST_N, H, C)

y_pred_test = pred_t.data[:, 0, 0].astype(np.float64)   # (TEST_N,)
y_true_test = X_val[TEST_START:TEST_START+TEST_N, 0, -HORIZON].astype(np.float64)
sigma_test  = pred_t.data[:, :, 0].std(axis=1).astype(np.float64) + 1e-6

lower, upper = conformal.predict_interval(y_pred_test, sigma_test)

coverage = float(np.mean((y_true_test >= lower) & (y_true_test <= upper)))
width    = float(np.mean(upper - lower))

print(f'\nConformal interval width:    {width:.4f}')
print(f'Coverage (target 90%):       {coverage*100:.1f}%')
print(f'Calibrated: {conformal.is_calibrated()}')

# ── Plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
x_ax = np.arange(TEST_N)
ax.fill_between(x_ax, lower, upper, alpha=0.3, color='steelblue', label='90% conformal interval')
ax.plot(x_ax, y_pred_test, 'b-',  label='Forecast (1-step ahead)', linewidth=1.5)
ax.plot(x_ax, y_true_test, 'r--', label='Actual',                  linewidth=1.5)
ax.set_title(f'NonStationary Conformal Prediction Intervals  '
             f'(coverage={coverage*100:.1f}%,  width={width:.3f})')
ax.set_xlabel('Test window index'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/conformal_intervals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 15: from_pretrained demo (after upload) ──────────────────────
# Once you've uploaded to HF Hub, anyone can do:

# pip install vulgaris huggingface_hub
# import vulgaris
# model = vulgaris.from_pretrained('keysparktech/vulgaris')

# Verify it works locally with the saved file:
loaded_model = vulgaris.from_pretrained(f'{OUTPUT_DIR}/{WEIGHT_NAME}.npz', config=cfg)

# Quick sanity check
test_x = vulgaris.Tensor(X_val[:4])
out1, _ = model(test_x)
out2, _ = loaded_model(test_x)
diff = float(np.abs(out1.data - out2.data).max())
print(f'Max output diff after reload: {diff:.2e}  (should be ~0)')
print('from_pretrained works correctly.')

In [ ]:
# ── Cell 16: Summary ──────────────────────────────────────────────────
print('=' * 60)
print('VULGARIS Pre-training Complete')
print('=' * 60)
print(f'  Model params      : {n_params:,} ({n_params/1e6:.2f}M)')
print(f'  Training windows  : {len(X_train):,}')
print(f'  Epochs trained    : {EPOCHS}')
print(f'  Final recon loss  : {history["recon"][-1]:.4f}')
print(f'  Final val loss    : {history["val_recon"][-1]:.4f}')
print()
print('Outputs saved:')
print('  /content/checkpoints/      — training checkpoints')
print('  /content/vulgaris-base-v1/ — final weights + config')
print('  /content/loss_curves.png   — training curves')
print('  /content/benchmark.png     — vs baselines')
print('  /content/anomaly_detection.png')
print('  /content/conformal_intervals.png')
print()
print('Hugging Face:')
print(f'  https://huggingface.co/{HF_REPO_ID}')
print()
print('Load anywhere with:')
print('  pip install vulgaris')
print(f'  model = vulgaris.from_pretrained("{HF_REPO_ID}")')